# EEG_07e — Build Grafi / Ipergrafi → Pruning → Tensori Prunati

Pipeline completa in 5 fasi:

| # | Fase | Output |
|---|------|--------|
| 1 | **Build** grafi + ipergrafi da CSV Paolo | `graph_{m}_k{k}.pt`, `hgraph_{m}_k{k}.pt` |
| 2 | **Comparison** plot grafo vs ipergrafo | figura |
| 3 | **Analisi connettività** per guida pruning | parametri PRUNE_* |
| 4 | **Build prunati** — rimuove canali deboli + archi sotto soglia | `*_drop*_thr*.pt` |
| 5 | **Sanity check** tensori prunati | stampa riepilogo |

**Input**: `data/raw_csv/training_set/PXXX_SYYY/parola_img.csv` (61 × 384, float)  
**Output**: `data/interim/graphs/*.pt`

---

### Consensus filter — Iacomi et al. 2026 (stesso dataset)

Iacomi et al. (2026) analizzano la connettività funzionale EEG sull'imagined speech usando PLV, wPLI, CPCCabs, CPCCim.
Trovano che mantenere solo gli archi presenti in **≥ 2/4 metriche** riduce i falsi positivi e produce grafi più stabili.

Risultati chiave (prior per la costruzione dei grafi):
- **Gamma** è la banda più funzionalmente integrata per IS (p=0.0015 vs beta)
- Tre pathway stabili: CL→FL,TL (delta, motor-language), TR→CL (gamma, auditory-to-motor), POL→CR (alpha, visual-spatial)
- Metriche consigliate per robustezza in basso SNR: CPCCabs > wPLI > PLV

Attiva con `CONSENSUS_FILTER = True` in cell config.

In [ ]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

# ── Fase 1: grafi / ipergrafi puliti ─────────────────────────
# Un file .pt per metrica, nessun pruning, nessun consensus.
# Aggiungi/rimuovi metriche qui; il loop in Fase 1 le itera tutte.
METHODS      = ["pcc", "plv", "wpli", "cpccabs", "cpccim"]
K_VALUES     = [6]
EDGE_THRESHOLD = 0.0   # 0 = nessuna soglia (grafi puliti)

# Ipergrafi: costruiti solo per le metriche selezionate
BUILD_HGNN   = True
METHODS_HGNN = ["pcc", "cpccabs"]   # subset di METHODS
K_HYPER      = 6

# Schema label
CLUSTER_SCHEME = "concr4"   # "concr4" | "ward4" | "sem5" | "pos4" | "raw110"

# Ricostruzione forzata
FORCE_REBUILD = False

# ── Fase B: consensus post-processing ───────────────────────
# Carica i .pt già buildati, mantieni archi in ≥ CONSENSUS_MIN metriche.
# Replica Iacomi et al. 2026 (PLV+wPLI+CPCCabs+CPCCim, min=2/4).
# Con PCC aggiunta come 5ª metrica: 3/5 equivale a ~60% consensus.
# Alternativa Iacomi-fedele: CONSENSUS_METHODS_B=["plv","wpli","cpccabs","cpccim"], CONSENSUS_MIN_B=2
CONSENSUS_METHODS_B = ["pcc", "plv", "wpli", "cpccabs", "cpccim"]
CONSENSUS_MIN_B     = 3   # ≥3/5

# ── Consensus inline (vecchia modalità — per confronto) ──────
# Costruisce il consensus edge_index ricalcolando le metriche per ogni trial.
# Più lento della Fase B ma non richiede i .pt singoli.
# ⚠️  PLV/wPLI sono O(N²) per trial → ore su dataset completo.
CONSENSUS_FILTER  = False
CONSENSUS_METHODS = ["pcc", "plv", "wpli"]
CONSENSUS_MIN     = 2

print("Config OK")
print(f"  Fase 1  : {METHODS}  k={K_VALUES}")
print(f"  Fase B  : consensus {CONSENSUS_MIN_B}/{len(CONSENSUS_METHODS_B)} "
      f"({CONSENSUS_METHODS_B})")
if CONSENSUS_FILTER:
    print(f"  Consensus inline: {CONSENSUS_METHODS}, min={CONSENSUS_MIN}")

In [ ]:
# ============================================================
# IMPORT E PATHS
# ============================================================

import os, sys, json
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import torch
torch.multiprocessing.set_sharing_strategy("file_system")

from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.data import Data

# --- Project root ---
project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

# --- Paths ---
CSV_ROOT   = project_root / "data" / "raw_csv" / "training_set"
GRAPHS_DIR = project_root / "data" / "interim" / "graphs"
CONFIGS    = project_root / "configs" / "label_schemes"
FIGURES    = project_root / "figures"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

N_CHANS   = 61
N_SAMPLES = 384

# --- Label mapping ---
with open(CONFIGS / "label2idx.json") as f:
    word2labelid = json.load(f)

labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(
    CLUSTER_SCHEME, project_root / "data" / "interim"
)

# --- Nomi canali (opzionale) ---
_eloc_candidates = [
    project_root / "data" / "interim" / "ebneuro.csv",
    Path("/mnt/c/Users/students/Desktop/Paolo/LM_Thesis/ebneuro.csv"),
]
CH_NAMES = [str(i) for i in range(N_CHANS)]
for _p in _eloc_candidates:
    if _p.exists():
        CH_NAMES = pd.read_csv(_p, sep=";", decimal=",")["labels"].tolist()[:N_CHANS]
        print(f"Canali da: {_p.name}")
        break

session_dirs = sorted(CSV_ROOT.iterdir())
print(f"Project root  : {project_root}")
print(f"Cartelle sess : {len(session_dirs)}")
print(f"Schema        : {CLUSTER_SCHEME} ({N_CLASSES} classi)")

In [ ]:
# ============================================================
# PARSING + HELPER
# ============================================================

def parse_folder(folder_name: str):
    """'P003_S002' → (3, 2)"""
    parts = folder_name.split("_")
    return int(parts[0][1:]), int(parts[1][1:])

def load_csv_trial(csv_path: Path) -> np.ndarray:
    """Legge CSV 61×384 senza header → float32 (61, 384)."""
    return pd.read_csv(csv_path, header=None).values.astype(np.float32)

def iter_all_trials():
    """
    Generator: itera tutte le cartelle PXXX_SYYY e tutti i CSV.
    Yields: (x_np, label_id, cluster_id, subj_id, sess_id)
    """
    for sess_dir in sorted(CSV_ROOT.iterdir()):
        if not sess_dir.is_dir():
            continue
        subj_id, sess_id = parse_folder(sess_dir.name)
        for csv_path in sorted(sess_dir.iterdir()):
            if csv_path.suffix != ".csv":
                continue
            word = csv_path.stem.replace("_img", "")
            if word not in word2labelid:
                continue
            label_id = word2labelid[word]
            if label_id not in labelid2cluster:
                continue
            x_np = load_csv_trial(csv_path)
            yield x_np, label_id, labelid2cluster[label_id], subj_id, sess_id

# Test parsing
test_dir = session_dirs[0]
test_csv = sorted(test_dir.iterdir())[0]
x_test   = load_csv_trial(test_csv)
assert x_test.shape == (N_CHANS, N_SAMPLES), f"Shape attesa (61,384), trovata {x_test.shape}"

n_total = sum(1 for _ in iter_all_trials())
print(f"Trial totali validi : {n_total}")
print(f"Shape CSV           : {x_test.shape}  ✅")

In [ ]:
# ============================================================
# FUNZIONI DI CONNETTIVITÀ
# ============================================================

def pcc_matrix(x_np: np.ndarray) -> np.ndarray:
    """Pearson |PCC| tra canali. Shape: (N, N)"""
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return pcc


def plv_matrix(x_np: np.ndarray) -> np.ndarray:
    """Phase Locking Value via Hilbert. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    phases = np.angle(hilbert(x_np, axis=1))
    plv = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            diff = phases[i] - phases[j]
            plv[i, j] = plv[j, i] = np.abs(np.mean(np.exp(1j * diff)))
    return plv


def wpli_matrix(x_np: np.ndarray) -> np.ndarray:
    """Weighted Phase Lag Index. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    analytic = hilbert(x_np, axis=1)
    wpli = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            cs = analytic[i] * np.conj(analytic[j])
            im = np.imag(cs)
            w  = np.abs(im)
            wpli[i, j] = wpli[j, i] = np.abs(np.mean(im * w)) / (np.mean(w) + 1e-9)
    return wpli


def cpcc_abs_matrix(x_np: np.ndarray) -> np.ndarray:
    """
    CPCCabs — |C_xy(f)| mediato sulle frequenze.
    C_xy = S_xy / sqrt(S_xx * S_yy)  (coerenza complessa).
    Robusto a basso SNR, immune ai bias di ampiezza.
    Shape: (N, N)
    """
    X    = np.fft.rfft(x_np, axis=1)                        # (N, F)
    Sxy  = X[:, None, :] * np.conj(X[None, :, :])           # (N, N, F)
    Sxx  = np.real(X * np.conj(X))                          # (N, F)
    denom = np.sqrt(Sxx[:, None, :] * Sxx[None, :, :]) + 1e-9
    C    = Sxy / denom                                       # (N, N, F) complessa
    mat  = np.abs(C).mean(axis=-1).astype(np.float32)
    np.fill_diagonal(mat, 0.0)
    return mat


def cpcc_im_matrix(x_np: np.ndarray) -> np.ndarray:
    """
    CPCCim — |Im(C_xy(f))| mediato sulle frequenze.
    La parte immaginaria della coerenza è zero per connettività istantanea
    (volume conduction) → immune agli artefatti di conduzione di volume.
    Valore assoluto: Im(C_xy) = -Im(C_yx) → serve |·| per grafo non diretto.
    Shape: (N, N)
    """
    X     = np.fft.rfft(x_np, axis=1)
    Sxy   = X[:, None, :] * np.conj(X[None, :, :])
    Sxx   = np.real(X * np.conj(X))
    denom = np.sqrt(Sxx[:, None, :] * Sxx[None, :, :]) + 1e-9
    C     = Sxy / denom
    mat   = np.abs(np.imag(C)).mean(axis=-1).astype(np.float32)
    np.fill_diagonal(mat, 0.0)
    return mat


# ── Dizionario principale ────────────────────────────────────
CONN_FN = {
    "pcc":     pcc_matrix,
    "plv":     plv_matrix,
    "wpli":    wpli_matrix,
    "cpccabs": cpcc_abs_matrix,
    "cpccim":  cpcc_im_matrix,
}


def knn_edge_index(matrix: np.ndarray, k: int,
                   threshold: float = 0.0) -> torch.LongTensor:
    """k-NN graph da matrice connettività. Restituisce edge_index (2, E)."""
    N = matrix.shape[0]
    rows, cols = [], []
    for i in range(N):
        row = matrix[i].copy(); row[i] = -1.0
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k = np.argsort(row)[-k:]
        for j in top_k:
            if row[j] > 0.0 or threshold == 0.0:
                rows += [i, j]; cols += [j, i]
    return torch.tensor([rows, cols], dtype=torch.long)


def consensus_edge_index(x_np: np.ndarray, k: int,
                          methods: list, min_consensus: int,
                          threshold: float = 0.0) -> torch.LongTensor:
    """
    Consensus inline — Iacomi et al. 2026.
    Mantieni arco (i,j) solo se presente nel k-NN di ≥ min_consensus metriche.
    Nota: ricalcola ogni metrica dal segnale raw → lento su dataset completo.
    Preferire la Fase B (consensus da .pt) per run complete.
    """
    N = x_np.shape[0]
    vote_matrix = np.zeros((N, N), dtype=np.int8)
    for method in methods:
        mat = CONN_FN[method](x_np)
        for i in range(N):
            row = mat[i].copy(); row[i] = -1.0
            if threshold > 0.0:
                row[row < threshold] = 0.0
            top_k = np.argsort(row)[-k:]
            for j in top_k:
                if row[j] > 0.0 or threshold == 0.0:
                    vote_matrix[i, j] += 1
                    vote_matrix[j, i] += 1
    src, dst = np.where(vote_matrix >= min_consensus)
    mask = src != dst
    return torch.tensor([src[mask].tolist(), dst[mask].tolist()], dtype=torch.long)


def hyperedge_index_fn(x_np: np.ndarray, k: int,
                       method: str = "pcc",
                       threshold: float = 0.0) -> torch.LongTensor:
    """
    Ipergrafo k-NN per-trial.
    Ogni nodo i è centro di un'iperedge con i + top-k vicini.
    """
    mat = CONN_FN[method](x_np)
    N   = mat.shape[0]
    vertex_list, edge_list = [], []
    for e_id in range(N):
        row = mat[e_id].copy()
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k   = np.argsort(row)[-k:]
        members = [e_id] + [j for j in top_k
                            if (row[j] > 0.0 or threshold == 0.0)]
        for v in members:
            vertex_list.append(v); edge_list.append(e_id)
    return torch.tensor([vertex_list, edge_list], dtype=torch.long)


print(f"Funzioni OK: {list(CONN_FN.keys())}")
print(f"  CPCCabs: |C_xy| (robustezza SNR)  |  CPCCim: |Im(C_xy)| (immune vol.cond.)")
if CONSENSUS_FILTER:
    print(f"  Consensus inline: {CONSENSUS_METHODS}, min={CONSENSUS_MIN}")

---
## Fase 1 — Build Grafi e Ipergrafi

In [ ]:
# ============================================================
# BUILD GRAPH TENSORS
# Output: data/interim/graphs/graph_{method}_k{k}.pt
#         oppure: graph_consensus{N}of{M}_k{k}.pt
# ============================================================

if CONSENSUS_FILTER:
    _n_m      = len(CONSENSUS_METHODS)
    _cons_tag = f"consensus{CONSENSUS_MIN}of{_n_m}"
    print(f"Modalità consensus: {CONSENSUS_METHODS}, min={CONSENSUS_MIN}/{_n_m}")
    print("⚠️  PLV/wPLI lenti — può richiedere ore su dataset completo.")

    for k in K_VALUES:
        out_path = GRAPHS_DIR / f"graph_{_cons_tag}_k{k}.pt"
        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip (esiste): {out_path.name}"); continue

        print(f"\nCostruendo {out_path.name}...")
        data_list, n_edges_log = [], []
        for x_np, label_id, cluster_id, subj_id, sess_id in tqdm(
                iter_all_trials(), total=n_total, desc=f"consensus k={k}"):
            ei = consensus_edge_index(x_np, k=k, methods=CONSENSUS_METHODS,
                                       min_consensus=CONSENSUS_MIN,
                                       threshold=EDGE_THRESHOLD)
            n_edges_log.append(ei.shape[1] // 2)
            data_list.append(Data(
                x=torch.tensor(x_np, dtype=torch.float32), edge_index=ei,
                y=torch.tensor(cluster_id, dtype=torch.long),
                label_id=torch.tensor(label_id, dtype=torch.long),
                subj=torch.tensor(subj_id, dtype=torch.long),
                sess=torch.tensor(sess_id, dtype=torch.long),
            ))
        torch.save(data_list, out_path)
        print(f"  ✅ {out_path.name} — {len(data_list)} grafi  "
              f"archi medi={np.mean(n_edges_log):.1f}")

else:
    for method in METHODS:
        for k in K_VALUES:
            out_path = GRAPHS_DIR / f"graph_{method}_k{k}.pt"
            if out_path.exists() and not FORCE_REBUILD:
                print(f"Skip (esiste): {out_path.name}"); continue

            print(f"\nCostruendo {out_path.name}...")
            conn_fn, data_list = CONN_FN[method], []
            for x_np, label_id, cluster_id, subj_id, sess_id in tqdm(
                    iter_all_trials(), total=n_total, desc=f"{method} k={k}"):
                matrix = conn_fn(x_np)
                ei     = knn_edge_index(matrix, k=k, threshold=EDGE_THRESHOLD)
                data_list.append(Data(
                    x=torch.tensor(x_np, dtype=torch.float32), edge_index=ei,
                    y=torch.tensor(cluster_id, dtype=torch.long),
                    label_id=torch.tensor(label_id, dtype=torch.long),
                    subj=torch.tensor(subj_id, dtype=torch.long),
                    sess=torch.tensor(sess_id, dtype=torch.long),
                ))
            torch.save(data_list, out_path)
            print(f"  ✅ {out_path.name} — {len(data_list)} grafi  "
                  f"x={data_list[0].x.shape}  ei={data_list[0].edge_index.shape}")

In [ ]:
# ============================================================
# BUILD HYPERGRAPH TENSORS
# Output: data/interim/graphs/hgraph_{method}_k{k}.pt
# ============================================================

if not BUILD_HGNN:
    print("BUILD_HGNN=False — skip.")
else:
    for method in METHODS_HGNN:
        for k in [K_HYPER]:
            out_path = GRAPHS_DIR / f"hgraph_{method}_k{k}.pt"
            if out_path.exists() and not FORCE_REBUILD:
                print(f"Skip (esiste): {out_path.name}"); continue

            print(f"\nCostruendo {out_path.name}...")
            data_list = []
            for x_np, label_id, cluster_id, subj_id, sess_id in tqdm(
                    iter_all_trials(), total=n_total,
                    desc=f"hgraph {method} k={k}"):
                he = hyperedge_index_fn(x_np, k=k, method=method,
                                        threshold=EDGE_THRESHOLD)
                data_list.append(Data(
                    x=torch.tensor(x_np, dtype=torch.float32),
                    hyperedge_index=he,
                    num_hyperedges=int(he[1].max().item()) + 1,
                    y=torch.tensor(cluster_id, dtype=torch.long),
                    label_id=torch.tensor(label_id, dtype=torch.long),
                    subj=torch.tensor(subj_id, dtype=torch.long),
                    sess=torch.tensor(sess_id, dtype=torch.long),
                ))
            torch.save(data_list, out_path)
            print(f"  ✅ {out_path.name} — {len(data_list)} ipergrafi  "
                  f"x={data_list[0].x.shape}  "
                  f"he={data_list[0].hyperedge_index.shape}  "
                  f"n_he={data_list[0].num_hyperedges}")

---
## Fase B — Consensus Post-Processing (da .pt già buildati)

Carica i grafi puliti di Fase 1 e mantiene solo gli archi presenti in
≥ `CONSENSUS_MIN_B` metriche su `CONSENSUS_METHODS_B`.

**Vantaggi rispetto al consensus inline:**
- Nessun ricalcolo dal segnale raw → ordini di grandezza più veloce
- Ogni metrica è già ottimizzata e salvata su disco
- Permette di cambiare `CONSENSUS_MIN_B` senza rieseguire Fase 1

**Prerequisito**: tutti i file `graph_{m}_k{k}.pt` in `CONSENSUS_METHODS_B` devono esistere.

Output: `graph_consensus{N}of{M}_k{k}.pt`

In [ ]:
# ============================================================
# FASE B — CONSENSUS POST-PROCESSING
# Carica i .pt di Fase 1, incrocia i k-NN edge sets per ogni trial.
# Arco (i,j) mantenuto se votato da ≥ CONSENSUS_MIN_B metriche.
# Zero ricalcoli dal segnale raw.
# ============================================================

import gc
from collections import Counter

for k in K_VALUES:
    # ── Verifica che tutti i file sorgente esistano ──────────
    src_paths = {}
    missing   = []
    for m in CONSENSUS_METHODS_B:
        p = GRAPHS_DIR / f"graph_{m}_k{k}.pt"
        if p.exists():
            src_paths[m] = p
        else:
            missing.append(p.name)

    if missing:
        print(f"⚠️  Fase B skip (k={k}) — file mancanti:")
        for f in missing:
            print(f"     → {f}")
        print("   Esegui prima Fase 1 per tutti i metodi in CONSENSUS_METHODS_B.")
        continue

    # ── Output path ─────────────────────────────────────────
    n_methods = len(CONSENSUS_METHODS_B)
    tag       = f"consensus{CONSENSUS_MIN_B}of{n_methods}"
    out_path  = GRAPHS_DIR / f"graph_{tag}_k{k}.pt"

    if out_path.exists() and not FORCE_REBUILD:
        print(f"Skip (esiste): {out_path.name}")
        continue

    print(f"\n[Fase B] {tag}  k={k}")
    print(f"  Metriche : {list(src_paths.keys())}")
    print(f"  Threshold: ≥{CONSENSUS_MIN_B}/{n_methods}")

    # ── Carica tutti i dataset in RAM ────────────────────────
    print("  Carico .pt in RAM...")
    datasets  = {m: torch.load(p, weights_only=False) for m, p in src_paths.items()}
    n_trials  = len(next(iter(datasets.values())))
    # Template: usa il primo dataset per x, y, label_id, subj, sess
    template  = next(iter(datasets.values()))
    print(f"  {n_trials} trial da combinare...")

    # ── Loop consensus ───────────────────────────────────────
    consensus_list = []
    n_edges_log    = []

    for i in tqdm(range(n_trials), desc=tag):
        vote = Counter()
        for dl in datasets.values():
            ei = dl[i].edge_index
            # Conta ogni coppia non-ordinata una volta per metrica
            for s, t in zip(ei[0].tolist(), ei[1].tolist()):
                if s < t:
                    vote[(s, t)] += 1

        kept = [(s, t) for (s, t), cnt in vote.items() if cnt >= CONSENSUS_MIN_B]
        n_edges_log.append(len(kept))

        if kept:
            ss = [s for s, t in kept] + [t for s, t in kept]  # aggiungi inverso
            dd = [t for s, t in kept] + [s for s, t in kept]  # per grafo non-dir.
            ei_new = torch.tensor([ss, dd], dtype=torch.long)
        else:
            # Fallback: nessun arco sopravvive al consensus
            ei_new = torch.zeros((2, 0), dtype=torch.long)

        d_ref = template[i]
        consensus_list.append(Data(
            x          = d_ref.x,          # segnale invariato
            edge_index = ei_new,
            y          = d_ref.y,
            label_id   = d_ref.label_id,
            subj       = d_ref.subj,
            sess       = d_ref.sess,
        ))

    # ── Salva + cleanup RAM ──────────────────────────────────
    torch.save(consensus_list, out_path)
    size_mb = out_path.stat().st_size / 1e6
    print(f"\n  ✅ Salvato: {out_path.name}  ({len(consensus_list)} grafi, {size_mb:.1f} MB)")
    print(f"     Archi medi: {np.mean(n_edges_log):.1f}  "
          f"min={min(n_edges_log)}  max={max(n_edges_log)}  "
          f"zero-edge trials: {sum(1 for e in n_edges_log if e == 0)}")

    del datasets, consensus_list
    gc.collect()

print("\n[Fase B] Completata.")

---
## Fase 2 — Comparison Plot (Grafo vs Ipergrafo)

In [ ]:
# ============================================================
# SANITY CHECK + COMPARISON PLOT grafo vs ipergrafo
# ============================================================

import networkx as nx

# --- Carica tutti i .pt base (non prunati) ---
graph_data, hgraph_data = {}, {}
print(f"=== File .pt in {GRAPHS_DIR} ===")
for pt_file in sorted(GRAPHS_DIR.glob("*.pt")):
    if "_drop" in pt_file.name or "_thr" in pt_file.name:
        continue   # salta prunati — li vediamo in Fase 5
    dl  = torch.load(pt_file, weights_only=False)
    d0  = dl[0]
    y_v = [d.y.item() for d in dl]
    n_subj = len(set(d.subj.item() for d in dl))
    is_hg  = pt_file.name.startswith("hgraph")
    if not is_hg:
        ei = d0.edge_index
        print(f"  {pt_file.name}: {len(dl)} grafi | x={d0.x.shape} | "
              f"ei={ei.shape} | subj={n_subj} | y=[{min(y_v)},{max(y_v)}]")
        graph_data[pt_file.stem] = dl
    else:
        he = d0.hyperedge_index
        print(f"  {pt_file.name}: {len(dl)} ipergrafi | x={d0.x.shape} | "
              f"he={he.shape} | n_he={d0.num_hyperedges} | "
              f"subj={n_subj} | y=[{min(y_v)},{max(y_v)}]")
        hgraph_data[pt_file.stem] = dl

if not graph_data and not hgraph_data:
    print("⚠️  Nessun .pt trovato — esegui prima Fase 1.")
else:
    print(f"\n✅ Sanity check OK — N_CHANS={N_CHANS}, schema={CLUSTER_SCHEME}")

# --- Comparison plot su trial campione ---
if graph_data and hgraph_data:
    g_key  = list(graph_data.keys())[0]
    hg_key = list(hgraph_data.keys())[0]
    d_g    = graph_data[g_key][0]
    d_hg   = hgraph_data[hg_key][0]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Comparison — trial campione  |  schema={CLUSTER_SCHEME}", fontsize=12)

    # 1. PCC heatmap
    x_np = d_g.x.numpy()
    pcc  = np.abs(np.corrcoef(x_np)); np.fill_diagonal(pcc, 0)
    im = axes[0].imshow(pcc, cmap="RdYlBu_r", vmin=0, vmax=0.5)
    axes[0].set_title("PCC matrice (trial campione)")
    axes[0].set_xlabel("Canale"); axes[0].set_ylabel("Canale")
    plt.colorbar(im, ax=axes[0])

    # 2. Grafo k-NN
    G = nx.Graph()
    G.add_nodes_from(range(d_g.x.shape[0]))
    ei = d_g.edge_index.numpy()
    for s, t in zip(ei[0], ei[1]):
        if s < t: G.add_edge(int(s), int(t))
    deg_g = [G.degree(n) for n in G.nodes()]
    pos   = nx.circular_layout(G)
    nx.draw_networkx(G, pos=pos, ax=axes[1],
                     node_size=80, node_color=deg_g, cmap="viridis",
                     with_labels=False, width=0.4, edge_color="gray")
    axes[1].set_title(f"Grafo k-NN ({g_key})\n"
                      f"{G.number_of_nodes()} nodi, {G.number_of_edges()} archi  "
                      f"grado medio={np.mean(deg_g):.1f}")

    # 3. Ipergrafo: istogramma dimensione iperedge
    he      = d_hg.hyperedge_index.numpy()
    n_nodes = d_hg.x.shape[0]
    n_he    = d_hg.num_hyperedges
    node_deg = np.bincount(he[0], minlength=n_nodes)
    he_size  = np.bincount(he[1], minlength=n_he)
    axes[2].bar(range(n_nodes), node_deg, color="steelblue", alpha=0.7)
    ax2t = axes[2].twinx()
    ax2t.hist(he_size, bins=20, color="orange", alpha=0.5)
    axes[2].set_xlabel("Nodo"); axes[2].set_ylabel("Grado nodo", color="steelblue")
    ax2t.set_ylabel("Freq. dim. iperedge", color="orange")
    axes[2].set_title(f"Ipergrafo ({hg_key})\n"
                      f"{n_nodes} nodi, {n_he} iperedge  dim.media={he_size.mean():.1f}")

    plt.tight_layout()
    fp = FIGURES / "eeg07e_comparison_graph_hgraph.png"
    fig.savefig(fp, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Plot salvato: {fp}")

---
## Fase 3 — Analisi Connettività (guida al pruning)

Calcola la connettività media su tutto il dataset per:
- identificare canali a bassa connettività (candidati alla rimozione)
- scegliere la soglia degli archi (`EDGE_THRESHOLD`)

In [ ]:
# ============================================================
# ANALISI CONNETTIVITÀ — tutti i trial
# Campiona 1 trial su VAL_SUBSAMPLE per l'istogramma (risparmio RAM)
# ============================================================

# wPLI normalizzato [0,1] per confronto diretto con PCC/PLV
def wpli_matrix_norm(x_np):
    m = wpli_matrix(x_np)
    mn, mx = m.min(), m.max()
    return (m - mn) / (mx - mn + 1e-9)

CONN_FN_ANALYSIS = {"pcc": pcc_matrix, "plv": plv_matrix, "wpli": wpli_matrix_norm}
VAL_SUBSAMPLE    = 10   # 1 trial su N per l'istogramma

if "plv" in METHODS or "wpli" in METHODS:
    print("⚠️  PLV/wPLI lenti — usa solo 'pcc' per analisi rapida.")

# --- Accumulatori ---
results = {m: {"sum": np.zeros((N_CHANS, N_CHANS)), "vals": [], "n": 0}
           for m in METHODS}

for i, (x_np, *_) in enumerate(tqdm(iter_all_trials(),
                                     total=n_total, desc="Analisi connettività")):
    for m in METHODS:
        mat = CONN_FN_ANALYSIS[m](x_np)
        results[m]["sum"] += mat
        results[m]["n"]   += 1
        if i % VAL_SUBSAMPLE == 0:
            results[m]["vals"].extend(mat[np.triu_indices(N_CHANS, k=1)].tolist())

for m in METHODS:
    results[m]["avg"]  = results[m]["sum"] / max(results[m]["n"], 1)
    results[m]["vals"] = np.array(results[m]["vals"])
    results[m]["conn"] = results[m]["avg"].sum(axis=1)

print(f"\n✅ Analisi — {results[METHODS[0]]['n']} trial, "
      f"campionato ogni {VAL_SUBSAMPLE}")

# --- Plot: heatmap + connettività per canale + distribuzione archi ---
n_methods = len(METHODS)
fig, axes = plt.subplots(3, n_methods, figsize=(6 * n_methods, 14))
if n_methods == 1:
    axes = axes[:, np.newaxis]
fig.suptitle(f"Analisi Connettività — {results[METHODS[0]]['n']} trial", fontsize=13)

for col, m in enumerate(METHODS):
    avg  = results[m]["avg"]
    conn = results[m]["conn"]
    vals = results[m]["vals"]
    mean_c, std_c = conn.mean(), conn.std()

    sns.heatmap(avg, ax=axes[0, col], cmap="RdYlBu_r",
                vmin=0, vmax=float(np.percentile(vals, 95)) if len(vals) else 1.0,
                xticklabels=False, yticklabels=False)
    axes[0, col].set_title(f"{m.upper()} — matrice media")

    bar_colors = [
        "red"       if c < mean_c - 2 * std_c else
        "orange"    if c < mean_c -     std_c else
        "steelblue"
        for c in conn
    ]
    axes[1, col].bar(range(N_CHANS), conn, color=bar_colors)
    axes[1, col].axhline(mean_c,              color="black",  ls="--", lw=1.2,
                         label=f"μ={mean_c:.3f}")
    axes[1, col].axhline(mean_c -     std_c,  color="orange", ls="--", lw=1.2,
                         label=f"μ-1σ={mean_c-std_c:.3f}")
    axes[1, col].axhline(mean_c - 2 * std_c,  color="red",    ls="--", lw=1.2,
                         label=f"μ-2σ={mean_c-2*std_c:.3f}")
    axes[1, col].set_xlabel("Canale"); axes[1, col].set_ylabel("Connettività totale")
    axes[1, col].set_title(f"{m.upper()} — per canale  🔴<μ-2σ  🟠<μ-1σ")
    axes[1, col].legend(fontsize=7)

    if len(vals):
        axes[2, col].hist(vals, bins=100, color="steelblue", alpha=0.7, edgecolor="none")
        for pct in [25, 50, 75]:
            thr = float(np.percentile(vals, pct))
            axes[2, col].axvline(thr, color="red", ls="--", lw=1.2,
                                 label=f"p{pct}={thr:.3f}")
        axes[2, col].set_xlabel(f"|{m.upper()}|")
        axes[2, col].set_ylabel("Frequenza")
        axes[2, col].set_title(f"{m.upper()} — distribuzione archi\n"
                               f"(guida EDGE_THRESHOLD, camp. 1/{VAL_SUBSAMPLE})")
        axes[2, col].legend(fontsize=7)

plt.tight_layout()
fp = FIGURES / "eeg07e_connectivity_analysis.png"
fig.savefig(fp, dpi=120, bbox_inches="tight")
plt.show()
print(f"Plot salvato: {fp}")

# --- Riepilogo testuale ---
for m in METHODS:
    conn  = results[m]["conn"]
    vals  = results[m]["vals"]
    mean_c, std_c = conn.mean(), conn.std()
    print(f"\n{'='*50}  {m.upper()}")
    print(f"  N trial : {results[m]['n']}")
    print(f"  μ={mean_c:.4f}  σ={std_c:.4f}")
    weak = np.where(conn < mean_c - 2 * std_c)[0]
    if len(weak):
        names_str = ", ".join(f"{CH_NAMES[i]}(idx={i})" for i in weak)
        print(f"  ⚠️  Canali deboli (<μ-2σ): {names_str}")
    else:
        print(f"  ✅ Nessun canale debole (<μ-2σ)")
    if len(vals):
        print("  Soglia archi consigliata:")
        for pct in [25, 50, 75]:
            thr  = float(np.percentile(vals, pct))
            kept = float((vals >= thr).mean() * 100)
            print(f"    p{pct:2d} = {thr:.4f}  → mantiene ~{kept:.0f}% archi")

---
## Fase 4 — Parametri Pruning

Auto-estratti dall'analisi sopra. Modifica `USE_PERCENTILE` e `SIGMA_THRESHOLD` se vuoi.
Poi esegui Fase 5.

In [ ]:
# ============================================================
# PARAMETRI PRUNING — auto-estratti da Fase 3
# ============================================================

if "results" not in globals() or "conn" not in results.get(METHODS[0], {}):
    raise RuntimeError("❌ Esegui prima Fase 3 (Analisi Connettività).")

REF_METHOD  = METHODS[0]
conn_ref    = results[REF_METHOD]["conn"]
vals_ref    = results[REF_METHOD]["vals"]
mean_c      = conn_ref.mean()
std_c       = conn_ref.std()

# --- 1. Canali da rimuovere: sotto μ - SIGMA_THRESHOLD * σ ---
SIGMA_THRESHOLD  = 2.0    # modifica: 1.0 | 1.5 | 2.0
weak_idx         = np.where(conn_ref < mean_c - SIGMA_THRESHOLD * std_c)[0]
PRUNE_CHANNEL_NAMES = [CH_NAMES[i] for i in weak_idx]

# --- 2. Soglia archi: percentile (0 = nessun pruning) ---
USE_PERCENTILE = 0        # modifica: 0 | 25 | 50 | 75
PRUNE_EDGE_THRESHOLD = (
    float(np.percentile(vals_ref, USE_PERCENTILE))
    if USE_PERCENTILE > 0 and len(vals_ref) > 0
    else 0.0
)

print("=" * 55)
print(f"PARAMETRI PRUNING (rif: {REF_METHOD.upper()})")
print("=" * 55)
if PRUNE_CHANNEL_NAMES:
    print(f"🔴 Canali rimossi (<μ-{SIGMA_THRESHOLD}σ): {PRUNE_CHANNEL_NAMES}")
else:
    print(f"✅ Nessun canale debole (soglia μ-{SIGMA_THRESHOLD}σ).")
if USE_PERCENTILE > 0:
    print(f"✂️  Soglia archi (p{USE_PERCENTILE}): {PRUNE_EDGE_THRESHOLD:.4f}")
else:
    print("✅ Nessun pruning archi (USE_PERCENTILE=0).")
print("=" * 55)

N_AFTER_PRUNE = N_CHANS - len(PRUNE_CHANNEL_NAMES)
print(f"\nCanali dopo pruning: {N_AFTER_PRUNE}/{N_CHANS}")

---
## Fase 5 — Build Grafi e Ipergrafi Prunati

Carica i tensori base (da Fase 1), applica:
- rimozione canali in `PRUNE_CHANNEL_NAMES`
- soglia archi `PRUNE_EDGE_THRESHOLD`

Output: `{nome_base}_drop{ch}_thr{val}.pt`

In [ ]:
# ============================================================
# BUILD GRAFI E IPERGRAFI PRUNATI
# ============================================================

if "PRUNE_CHANNEL_NAMES" not in globals():
    raise RuntimeError("❌ Esegui prima Fase 4 (Parametri Pruning).")

# Nessun pruning attivo?
if not PRUNE_CHANNEL_NAMES and PRUNE_EDGE_THRESHOLD == 0.0:
    print("⚠️  Nessun pruning attivo — imposta SIGMA_THRESHOLD o USE_PERCENTILE in Fase 4.")
else:
    # --- Indici canali da tenere ---
    _drop_idx  = set()
    for _name in PRUNE_CHANNEL_NAMES:
        if _name in CH_NAMES:
            _drop_idx.add(CH_NAMES.index(_name))
        else:
            print(f"⚠️  Canale '{_name}' non trovato — saltato")
    _keep    = [i for i in range(N_CHANS) if i not in _drop_idx]
    N_PRUNED = len(_keep)

    print(f"Canali rimossi : {[CH_NAMES[i] for i in _drop_idx] or 'nessuno'}")
    print(f"Canali rimasti : {N_PRUNED}/{N_CHANS}")
    print(f"Edge threshold : {PRUNE_EDGE_THRESHOLD}")

    # --- Suffisso nome file ---
    _suf_ch  = ("_drop" + "_".join(PRUNE_CHANNEL_NAMES)) if PRUNE_CHANNEL_NAMES else ""
    _suf_thr = f"_thr{PRUNE_EDGE_THRESHOLD:.3f}" if PRUNE_EDGE_THRESHOLD > 0 else ""
    _suffix  = _suf_ch + _suf_thr

    for pt_file in sorted(GRAPHS_DIR.glob("*.pt")):
        # Salta file già prunati
        if "_drop" in pt_file.name or "_thr" in pt_file.name:
            continue

        is_hgraph = pt_file.name.startswith("hgraph")
        _k        = int(pt_file.stem.split("_k")[-1]) if "_k" in pt_file.stem else K_HYPER
        out_path  = GRAPHS_DIR / (pt_file.stem + _suffix + ".pt")

        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip (esiste): {out_path.name}")
            continue

        print(f"\nPruning {pt_file.name} → {out_path.name} ...")
        dl = torch.load(pt_file, weights_only=False)
        pruned_list = []

        for d in tqdm(dl, desc=out_path.name, leave=False):
            x_np     = d.x.numpy()              # (N_CHANS, 384)
            x_pruned = x_np[_keep, :]           # (N_PRUNED, 384)

            if not is_hgraph:
                # Grafo
                if PRUNE_EDGE_THRESHOLD > 0.0:
                    _mat = pcc_matrix(x_pruned)
                    _ei  = knn_edge_index(_mat, k=_k,
                                          threshold=PRUNE_EDGE_THRESHOLD)
                else:
                    _idx_map = {old: new for new, old in enumerate(_keep)}
                    _old_ei  = d.edge_index.numpy()
                    _mask    = [
                        (int(s) in _idx_map and int(t) in _idx_map)
                        for s, t in zip(_old_ei[0], _old_ei[1])
                    ]
                    _ei = torch.tensor(
                        [[_idx_map[int(s)] for s, ok in zip(_old_ei[0], _mask) if ok],
                         [_idx_map[int(t)] for t, ok in zip(_old_ei[1], _mask) if ok]],
                        dtype=torch.long
                    )
                pruned_list.append(Data(
                    x=torch.tensor(x_pruned, dtype=torch.float32),
                    edge_index=_ei,
                    y=d.y, label_id=d.label_id, subj=d.subj, sess=d.sess,
                ))
            else:
                # Ipergrafo: ricalcola hyperedge_index sui canali prunati
                _he = hyperedge_index_fn(x_pruned, k=_k,
                                         threshold=PRUNE_EDGE_THRESHOLD)
                pruned_list.append(Data(
                    x=torch.tensor(x_pruned, dtype=torch.float32),
                    hyperedge_index=_he,
                    num_hyperedges=int(_he[1].max()) + 1,
                    y=d.y, label_id=d.label_id, subj=d.subj, sess=d.sess,
                ))

        torch.save(pruned_list, out_path)
        d0 = pruned_list[0]
        if not is_hgraph:
            print(f"  ✅ {out_path.name}: {len(pruned_list)} grafi  "
                  f"x={d0.x.shape}  ei={d0.edge_index.shape}")
        else:
            print(f"  ✅ {out_path.name}: {len(pruned_list)} ipergrafi  "
                  f"x={d0.x.shape}  he={d0.hyperedge_index.shape}  "
                  f"n_he={d0.num_hyperedges}")

---
## Fase 6 — Sanity Check Tensori Prunati

In [ ]:
# ============================================================
# SANITY CHECK — tutti i .pt in GRAPHS_DIR
# Stampa riepilogo base vs prunati a confronto
# ============================================================

all_pt = sorted(GRAPHS_DIR.glob("*.pt"))
if not all_pt:
    print("⚠️  Nessun .pt trovato in", GRAPHS_DIR)
else:
    print(f"{'File':55s}  {'N':>6}  {'x.shape':>12}  {'tipo':>10}  {'canali':>7}")
    print("-" * 100)
    for pt_file in all_pt:
        dl  = torch.load(pt_file, weights_only=False)
        d0  = dl[0]
        is_pruned = "_drop" in pt_file.name or "_thr" in pt_file.name
        is_hg     = pt_file.name.startswith("hgraph")
        tipo      = "hgraph" if is_hg else "graph"
        tag       = "[PRUNATO]" if is_pruned else "[BASE]   "
        n_ch      = d0.x.shape[0]
        if is_hg:
            shape_str = f"{tuple(d0.x.shape)!s}  he={d0.hyperedge_index.shape[1]}"
        else:
            shape_str = f"{tuple(d0.x.shape)!s}  ei={d0.edge_index.shape[1]}"
        print(f"{tag} {pt_file.name:50s}  {len(dl):>6}  {shape_str:>20}  {tipo:>10}  {n_ch:>7}")

    # Verifica coerenza prunati
    print("\n--- Verifiche ---")
    errors = 0
    for pt_file in all_pt:
        if not ("_drop" in pt_file.name or "_thr" in pt_file.name):
            continue
        dl = torch.load(pt_file, weights_only=False)
        n_ch_actual = dl[0].x.shape[0]
        if n_ch_actual >= N_CHANS:
            print(f"⚠️  {pt_file.name}: canali={n_ch_actual} (atteso < {N_CHANS})")
            errors += 1
        else:
            print(f"✅ {pt_file.name}: canali={n_ch_actual}/{N_CHANS}  "
                  f"({N_CHANS - n_ch_actual} rimossi)")

    if errors == 0:
        print("\n✅ Tutti i tensori prunati sono consistenti.")
    else:
        print(f"\n❌ {errors} file con problemi — ricontrolla Fase 4-5.")